# FAOSTAT food trade data

🚧 **Work in progress**

## Introduction

This is a technical companion to the custom food-trade visualization ***(link).
Here we analyse the content and limitations of FAOSTAT's *Detailed Trade Matrix* (TM) dataset.

The TM dataset contains _item flows_.
An item is a food product, either a primary commodity such as wheat or bananas, or a processed product such as meat (fish products are not included in the dataset).
A flow is a trade exchange between two countries, namely an import or an export.

Each flow is measured as a quantity (in tonnes, or number of animals) and as a monetary value (in dollars).
Here we will analyse quantities of traded food (in tonnes).

An entry in the data corresponds to a trade flow reported by a certain country and year, with a given partner country.
For example, in 2023, Brazil reports exporting ~4 million tonnes of soya beans to Argentina.

Ideally, one would expect that item flows would be reported both ways, and that both countries would agree on the traded amount.
Indeed, there's an entry for 2023, where Argentina reports importing ~4 million tonnes of soya beans from Brazil.

However, after inspecting the data, it turns out that this is not always the case.
- Item flows are often reported only one way.
- When item flows are reported both ways, the quantity often diverges.

In the following sections we will inspect data coverage and reporting issues.

## Initial setup

To be able to run the code, there are some libraries you may need to install.

In [ ]:
# Whether you run this code in your local computer or on Google Colab, you first need to install the required dependencies:
# * owid-catalog: a library that lets you explore and load datasets from OWID.
# * plotly: a library that lets you create interactive visualizations.
%pip install --upgrade owid-catalog plotly > /dev/null 2>&1

Import necessary libraries.

In [54]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from owid.catalog import Dataset

from etl.paths import DATA_DIR

ds = Dataset(DATA_DIR / "garden" / "faostat" / "2026-05-07" / "faostat_tm")
tb = ds.read("faostat_tm", safe_types=False)

ds_qcl = Dataset(DATA_DIR / "garden" / "faostat" / "2026-02-25" / "faostat_qcl")
tb_qcl = ds_qcl.read("faostat_qcl", safe_types=False)

## Basic data preparation

The TM dataset has ~52 million entries.

We select only rows of item flow quantities (in tonnes); then the dataset contains ~26 million entries.

We also drop rows of self-trade, where reporter and partner countries coincide (which are just a ~0.1% of the data).

Around 12% of rows correspond to trade of zero tonnes (which we keep in the data).

In [116]:
# Total number of entries
# len(tb)

# Basic data preparation: restrict to quantity reports in tonnes (this drops the
# alternate-unit rows used for live animals — 'An', '1000 An', 'No' — and the
# value-in-USD rows),
tb = tb[(tb["unit"] == "t")].reset_index(drop=True)
# len(tb) * 1e-6

# Drop self-trade rows (~0.1% of the data, where reporter
# and partner are the same country).
cats = sorted(set(tb["reporter_country"].cat.categories) | set(tb["partner_country"].cat.categories))
tb = tb[(tb["unit"] == "t") & (tb["reporter_country"].cat.set_categories(cats).cat.codes != tb["partner_country"].cat.set_categories(cats).cat.codes)].reset_index(drop=True)
# len(tb) * 1e-6

# Share of rows with zero tonnes reported.
# 100 * len(tb[tb["value"] == 0]) / len(tb)

## Year coverage

We would ideally show food trade figures for the most recent year available in the data.
However, the latest year is may be affected by incomplete reporting.

The following chart helps us decide which year to choose.
It shows the number of rows, as well as the number of distinct reporting countries, for each year in the dataset.

In [59]:
def plot_coverage(tb):
    """Show two bars per year: number of rows and number of distinct
    reporting countries. When both drop together a year is partially
    reported; when only rows drop it might be due to an actual trade contraction."""
    grouped = tb.groupby("year", observed=True)
    rows = grouped.size().sort_index()
    reporters = grouped["reporter_country"].nunique().sort_index()
    years = rows.index.astype(int).tolist()

    fig = go.Figure()
    fig.add_bar(x=years, y=rows.values, name="Rows", yaxis="y1", opacity=0.8)
    fig.add_bar(x=years, y=reporters.values, name="Distinct reporters", yaxis="y2", opacity=0.8)
    fig.update_layout(
        title="Yearly data coverage",
        xaxis=dict(title="Year"),
        yaxis=dict(title="Number of rows", side="left"),
        yaxis2=dict(title="Number of distinct reporters", side="right", overlaying="y", showgrid=False),
        barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

plot_coverage(tb)

We can see that the latest year (2024) shows a significant drop in coverage with respect to the second latest.

So, in our custom visualization, and in the rest of the analysis, we will focus on the latest year with a reasonably complete coverage, namely 2023.

In [107]:
# Year to use for year-specific analyses from now on.
YEAR = 2023

## Reporting coverage by year

As mentioned before, the TM dataset relies on quantities reported by countries.

Ideally, if a country A reports the export of a certain item to a country B, one would expect country B to report the import of the same quantity of the item from country A.
But how often are item flows reported both ways?
And how much disagreement is there between countries' reported quantities?

For each item flow (an item going from a specific country to another) in a year, we count how many times that flow was reported both ways ("matched") or only one way ("exporter-only" or "importer-only").
The result is shown in the following chart.

In [60]:
def plot_reporting_coverage_by_year(tb):
    """Stacked bar chart per year showing what fraction of bilateral flows
    are reported by both sides (matched), only by the exporter, or only by
    the importer."""
    qty = tb[tb["element"].isin(["Export quantity", "Import quantity"])][
        ["reporter_country", "partner_country", "item_code", "year", "element"]
    ].copy()
    for col in ("reporter_country", "partner_country"):
        qty[col] = qty[col].astype(str)

    exp_keys = qty.loc[
        qty["element"] == "Export quantity",
        ["reporter_country", "partner_country", "item_code", "year"],
    ].drop_duplicates()
    imp_keys = (
        qty.loc[
            qty["element"] == "Import quantity",
            ["reporter_country", "partner_country", "item_code", "year"],
        ]
        .rename(columns={"reporter_country": "partner_country", "partner_country": "reporter_country"})
        .drop_duplicates()
    )

    merged = exp_keys.merge(
        imp_keys, how="outer", indicator=True,
        on=["reporter_country", "partner_country", "item_code", "year"],
    )
    by_year = (
        merged.groupby("year", observed=True)["_merge"]
        .value_counts(normalize=True)
        .unstack(fill_value=0.0)
        .rename(columns={"both": "matched", "left_only": "exporter-only", "right_only": "importer-only"})
    )
    by_year = by_year[["matched", "exporter-only", "importer-only"]].reset_index()
    long = by_year.melt(id_vars="year", var_name="status", value_name="share")

    fig = px.bar(
        long, x="year", y="share", color="status",
        title="Reporting coverage",
        labels={"year": "Year", "share": "Share of (reporter, partner, item) tuples"},
        category_orders={"status": ["matched", "exporter-only", "importer-only"]},
    )
    fig.update_layout(barmode="stack", yaxis_tickformat=".0%")
    fig.show()

plot_reporting_coverage_by_year(tb)

We can see that the reporting inconsistency is quite severe.
Most item flows are reported only one way (only the importer, or only the exporter).

In 2023, only around 40% of item exchanges were reported both ways.
In other words, 60% of all (exporter, importer, item) flows show up in only one side's records.

In [117]:
# Simple code to double-check that result for 2023 alone.
exporter_reports = tb[(tb["year"] == YEAR) & (tb["element"] == "Export quantity")].rename(columns={"reporter_country": "exporter", "partner_country": "importer"})
importer_reports = tb[(tb["year"] == YEAR) & (tb["element"] == "Import quantity")].rename(columns={"reporter_country": "importer", "partner_country": "exporter"})
exporter_reports_set = set(exporter_reports["exporter"].astype("string") + exporter_reports["importer"].astype("string") + exporter_reports["item"].astype("string"))
importer_reports_set = set(importer_reports["exporter"].astype("string") + importer_reports["importer"].astype("string") + importer_reports["item"].astype("string"))
# Complete set of item flows reported (by importers, exporters, or both).
all_flows_set = importer_reports_set | exporter_reports_set
# Share of all flows that are reported by importers only.
# 100 * len(importer_reports_set - exporter_reports_set) / len(all_flows_set)

# Share of all flows that are reported by exporters only.
# 100 * len(exporter_reports_set - importer_reports_set) / len(all_flows_set)

# Share of all flows that are reported by both importers and exporters:
# 100 * len(exporter_reports_set & importer_reports_set) / len(all_flows_set)

Now, among those exchanges that were reported both ways, how well did they agree with each other?

## Quantity agreement

For matched flows, how closely do the two reported quantities agree? The
agreement ratio is `min(exp_qty, imp_qty) / max(exp_qty, imp_qty)` —
100% means a perfect match, 50% means one side reports twice the other.
The chart bins each reporter's matched flows into five bands and shows
the worst reporters at the top.

In [13]:
def plot_quantity_mismatch_by_reporter(tb, year=None, top_n=20, include_unmatched=False):
    """Stacked horizontal bar showing how bilateral flows distribute across
    quantity-agreement bands, for the top-N reporting countries.

    Agreement = min(exp_qty, imp_qty) / max(exp_qty, imp_qty). 100% = perfect
    match, 50% = one side reports 2x the other, etc. Bins into five bands
    (red = poor, green = near-perfect). When include_unmatched=True, flows
    reported by only one side land in a sixth Unmatched band.
    """
    if year is None:
        rows_per_year = tb.groupby("year", observed=True).size()
        year = int(rows_per_year[rows_per_year >= 0.9 * rows_per_year.max()].index.max())

    qty = tb[(tb["year"] == year) & tb["element"].isin(["Export quantity", "Import quantity"])][
        ["reporter_country", "partner_country", "item_code", "element", "value"]
    ].copy()
    for col in ("reporter_country", "partner_country"):
        qty[col] = qty[col].astype(str)

    exp = qty.loc[
        qty["element"] == "Export quantity",
        ["reporter_country", "partner_country", "item_code", "value"],
    ].rename(columns={"value": "exp_qty"})
    imp = qty.loc[
        qty["element"] == "Import quantity",
        ["reporter_country", "partner_country", "item_code", "value"],
    ].rename(columns={"reporter_country": "partner_country", "partner_country": "reporter_country", "value": "imp_qty"})

    how = "outer" if include_unmatched else "inner"
    merged = exp.merge(imp, on=["reporter_country", "partner_country", "item_code"], how=how)
    merged["exp_qty"] = merged["exp_qty"].fillna(0)
    merged["imp_qty"] = merged["imp_qty"].fillna(0)
    if include_unmatched:
        merged = merged[(merged["exp_qty"] > 0) | (merged["imp_qty"] > 0)].copy()
    else:
        merged = merged[(merged["exp_qty"] > 0) & (merged["imp_qty"] > 0)].copy()

    exp_arr = merged["exp_qty"].to_numpy()
    imp_arr = merged["imp_qty"].to_numpy()
    max_arr = np.maximum(exp_arr, imp_arr)
    merged["agreement"] = np.minimum(exp_arr, imp_arr) / max_arr

    matched_band_labels = ["<25%", "25-50%", "50-75%", "75-90%", ">=90%"]
    matched_band_colors = ["#d73027", "#fc8d59", "#fee08b", "#91cf60", "#1a9850"]
    band = pd.cut(merged["agreement"], bins=[-0.001, 0.25, 0.50, 0.75, 0.90, 1.001], labels=matched_band_labels)
    if include_unmatched:
        unmatched_mask = (merged["exp_qty"] == 0) | (merged["imp_qty"] == 0)
        band = band.cat.add_categories("Unmatched")
        band[unmatched_mask] = "Unmatched"
        band_labels = ["Unmatched"] + matched_band_labels
        band_colors = ["#67000d"] + matched_band_colors
    else:
        band_labels = matched_band_labels
        band_colors = matched_band_colors
    merged["band"] = band

    top = merged["reporter_country"].value_counts().head(top_n).index.tolist()
    sub = merged[merged["reporter_country"].isin(top)]
    shares = sub.groupby(["reporter_country", "band"], observed=True).size().unstack(fill_value=0)
    shares = shares.div(shares.sum(axis=1), axis=0)[band_labels]
    shares = shares.sort_values(">=90%", ascending=True)

    long = shares.reset_index().melt(id_vars="reporter_country", var_name="band", value_name="share")
    title = (
        f"Quantity agreement (matched flows + unmatched as 0%) in {year}" if include_unmatched
        else f"Quantity agreement among matched flows in {year}"
    )
    x_label = "Share of all flows" if include_unmatched else "Share of matched flows"
    fig = px.bar(
        long, x="share", y="reporter_country", color="band", orientation="h",
        title=title,
        labels={"share": x_label, "reporter_country": "Reporter"},
        category_orders={"band": band_labels, "reporter_country": shares.index.tolist()},
        color_discrete_sequence=band_colors,
    )
    fig.update_layout(barmode="stack", xaxis_tickformat=".0%")
    fig.show()

plot_quantity_mismatch_by_reporter(tb, year=2023, top_n=20, include_unmatched=False)

For Brazil, for example, we see that only around 40% of those matched flows agreed with each other well (the reported quantity matched between 75 and 100%).

This is surprisingly bad; but maybe the mismatches are not as bad when we select the important items.

The following chart is identical to the previous, but it includes flows reported only one-way, in a dedicated "Unmatched" band. 
This way we can visualize both reporting coverage and quantity-agreement issues in one visualization.

In [14]:
plot_quantity_mismatch_by_reporter(tb, year=2023, top_n=20, include_unmatched=True)

TODO: How do the previous charts change for the list of items selected?

## List of item flows to include

The dataset has over 500 items.
Instead of including all of them, we select a curated list of items.

TODO: inspect list and decide which ones to include.

### Reconciling disagreeing reports

When both countries report the same flow with different quantities (which happens often, as the previous charts show), we need a single number to show in the visualization.

Our criterion:
- **Only one side reports the flow** → use that side's number.
- **Both sides report the flow** → use the **geometric mean** of the two quantities.

The geometric mean is the natural average for trade flows because they span many orders of magnitude (a country can trade a few tonnes or millions of tonnes of an item with another country), and it treats over- and under-reporting **symmetrically in log space**.

For example: in 2023 Ukraine reports exporting 2,268,088 tonnes of wheat to Romania, but Romania only reports importing 207,743 tonnes — a factor of ~11 apart. The geometric mean is √(2,268,088 × 207,743) ≈ **686,371 tonnes**. Both reports sit at a factor of ~3.3 from this central value (one ~3.3× above, one ~3.3× below). The arithmetic mean (≈ 1,237,916 tonnes) would instead sit closer to the larger of the two reports.

In [119]:
# tb[(tb["reporter_country"] == "Ukraine") & (tb["partner_country"] == "Romania") & (tb["element"] == "Export quantity") & (tb["item"] == "Wheat") & (tb["year"] == YEAR)]
# tb[(tb["reporter_country"] == "Romania") & (tb["partner_country"] == "Ukraine") & (tb["element"] == "Import quantity") & (tb["item"] == "Wheat") & (tb["year"] == YEAR)]


## Including production and domestic supply
Trade figures are more informative in relative terms:

* exports as a share of what a country produces, and
* imports as a share of what a country consumes domestically.

In other words, we would ideally show exports as a share of production, and imports as a share of domestic supply.

However, production and domestic supply are not available in the TM dataset.
So we need to combine this with other FAOSTAT datasets.

### Exports as a share of production
FAOSTAT's Crops and livestock products (QCL) dataset contains production quantity of each item, for each country and year.
Thankfully, QCL and TM datasets use the same item codes (a number that uniquely identifies a FAOSTAT item).
So it is in principle possible to combine both datasets to have production data in our food trade visualizations.

However, only about 45% of items in the TM dataset are informed in the QCL dataset.
Specifically for 2023, we have production data for only 53% of traded items.

In [118]:
item_codes_tm = set(tb[tb["year"] == YEAR]["item_code"])
item_codes_qcl = set(tb_qcl[tb_qcl["year"] == YEAR]["item_code"].astype(int))
# 100 * len(item_codes_tm - item_codes_qcl) / len(item_codes_tm)

Among traded items for which there is production data, is it accurate to express exports as a share of production, given the reporting issues mentioned above?
Provided that the visualization clearly states that it shows *reported* flows, the numbers are accurate.
If a certain item export is underreported, one will see that a country *reports the exports of a small share of its production*.
This is not the same as claiming that a country *exports a small share of its production*.

So, as long as the reporting limitation is mentioned, there are no correctness issues.

### Imports as a share of domestic supply

Domestic supply is not available in either TM or QCL datasets.
We can easily estimate it as Production + Imports - Exports.
This estimate is possible for only those traded items for which we have production data.
Now, is it accurate to express imports as a share of domestic supply, given the reporting issues?

Suppose a country underreports their exports of a certain item.
Now, not only those exports will be an underestimate of the true amount, but also domestic supply will be overestimated (since exports will not be properly subtracted from the amount of production and imports).

So, it is unclear if we should include the estimated domestic supply.

TODO: Discuss this issue.

Could we source domestic supply data from a different dataset?

Instead of estimating domestic supply, we could get this data from FAOSTAT's Food Balances dataset (FBS).
Indeed, the FBS dataset has official figures for production, domestic supply, imports, and exports.
However, mapping FBS items to QCL or TM items is not straightforward.

For example, FBS' item "Cocoa Beans and products" is a group of various items.
As its metadata says:
`Default composition: 661 Cocoa, beans, 662 Cocoa, paste, 665 Cocoa, powder & cake, 666 Chocolate products nes`

There's [a Nature paper](https://www.nature.com/articles/s41597-025-05137-y) that does a proper disaggregation of FBS commodities (the data ends in 2022).
We could consider using this dataset to map FBS items to those in the TM dataset.

For now, I'll consider this approach as out of the scope of our project.